In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Replicator–Documentation Evaluator

## Task: Evaluate whether the replicator's documentation faithfully reproduces the results and conclusions of the original experiment.

### Inputs:
- **Original repo:** `/net/scratch2/smallyan/rome_eval`
- **Replication outputs:** `/net/scratch2/smallyan/rome_eval/evaluation/replications`

In [2]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")

CUDA available: True
GPU device: NVIDIA A100 80GB PCIe
Number of GPUs: 1


In [3]:
# Define paths
original_repo = '/net/scratch2/smallyan/rome_eval'
replication_dir = '/net/scratch2/smallyan/rome_eval/evaluation/replications'
output_dir = '/net/scratch2/smallyan/rome_eval/evaluation/replication_eval'

# Check if paths exist
print(f"Original repo exists: {os.path.exists(original_repo)}")
print(f"Replication dir exists: {os.path.exists(replication_dir)}")

# List contents of original repo
print("\n--- Original repo contents ---")
if os.path.exists(original_repo):
    for item in os.listdir(original_repo):
        print(f"  {item}")

Original repo exists: True
Replication dir exists: True

--- Original repo contents ---
  util
  hparams
  rome
  globals.yml
  .git
  CodeWalkthrough.md
  dsets
  .gitignore
  experiments
  plan.md
  CITATION.cff
  evaluation
  documentation.pdf
  LICENSE
  notebooks
  baselines
  data
  scripts
  .gitattributes


In [4]:
# List contents of replication directory
print("--- Replication directory contents ---")
if os.path.exists(replication_dir):
    for item in os.listdir(replication_dir):
        full_path = os.path.join(replication_dir, item)
        if os.path.isdir(full_path):
            print(f"  [DIR] {item}")
            for sub_item in os.listdir(full_path):
                print(f"        {sub_item}")
        else:
            print(f"  {item}")

--- Replication directory contents ---
  [DIR] 2026-01-11_12-20-56
        test_success.txt
        replication_script.py.py
        replication.ipynb
        self_replication_evaluation.json
        documentation_replication.md
        README.md
        evaluation_replication.md
        test_replication.py


In [5]:
# Find the documentation files
import glob

# Check for documentation.md in original repo
original_doc_md = os.path.join(original_repo, 'documentation.md')
original_doc_pdf = os.path.join(original_repo, 'documentation.pdf')

print(f"Original documentation.md exists: {os.path.exists(original_doc_md)}")
print(f"Original documentation.pdf exists: {os.path.exists(original_doc_pdf)}")

# Look for markdown files in the original repo
print("\n--- Markdown files in original repo ---")
md_files = glob.glob(os.path.join(original_repo, '*.md'))
for f in md_files:
    print(f"  {os.path.basename(f)}")

Original documentation.md exists: False
Original documentation.pdf exists: True

--- Markdown files in original repo ---
  CodeWalkthrough.md
  plan.md


In [6]:
# Read the replicated documentation
replication_timestamp = '2026-01-11_12-20-56'
replicated_doc_path = os.path.join(replication_dir, replication_timestamp, 'documentation_replication.md')

print(f"Replicated documentation path: {replicated_doc_path}")
print(f"Exists: {os.path.exists(replicated_doc_path)}")

if os.path.exists(replicated_doc_path):
    with open(replicated_doc_path, 'r') as f:
        replicated_doc = f.read()
    print("\n--- Replicated Documentation Content ---")
    print(replicated_doc)

Replicated documentation path: /net/scratch2/smallyan/rome_eval/evaluation/replications/2026-01-11_12-20-56/documentation_replication.md
Exists: True

--- Replicated Documentation Content ---
# Causal Tracing Replication Documentation

## Goal

Replicate the causal tracing experiment from "Locating and Editing Factual Associations in GPT" to identify which hidden states in transformer language models are causally responsible for factual predictions. The experiment uses a double-intervention method to trace the flow of factual information through the network.

## Research Question

Where in a transformer language model (GPT-2 XL) are factual associations stored and recalled? Specifically:
1. Which layers contain the critical information?
2. Which token positions are most important?
3. Are MLP modules or attention modules more causally important?

## Data

**Dataset:** Known Facts Dataset (`known_1000.json`)
- 1,000 factual statements in the form of prompts
- Each entry contains:
  - `pr

In [7]:
# Read the original documentation.pdf - we need to extract text from it
# First, let's also look at other relevant files to understand the original results
# Check if there are READMEs or other documentation

# Also check for notebooks which might have original results
notebooks_path = os.path.join(original_repo, 'notebooks')
print("--- Notebooks ---")
if os.path.exists(notebooks_path):
    for item in os.listdir(notebooks_path):
        print(f"  {item}")
        
# Check experiments folder
experiments_path = os.path.join(original_repo, 'experiments')
print("\n--- Experiments ---")
if os.path.exists(experiments_path):
    for item in os.listdir(experiments_path):
        print(f"  {item}")

--- Notebooks ---
  experiments
  globals.yml
  average_causal_effects.ipynb
  baselines
  dsets
  causal_trace.ipynb
  causal_trace_frozen_mlp_attn.ipynb
  vis
  util
  hparams
  rome.ipynb
  rome

--- Experiments ---
  evaluate.py
  summarize.py
  __init__.py
  causal_trace.py
  sweep.py
  py
  __pycache__


In [8]:
# Let's try to read the PDF using pdfplumber or PyPDF2
import subprocess

# First check if we have PDF reading capability
try:
    import pdfplumber
    print("pdfplumber is available")
except ImportError:
    print("pdfplumber not available, trying PyPDF2...")
    try:
        import PyPDF2
        print("PyPDF2 is available")
    except ImportError:
        print("PyPDF2 not available, trying pypdf...")
        try:
            import pypdf
            print("pypdf is available")
        except ImportError:
            print("No PDF library available")

pdfplumber is available


In [9]:
# Read the original documentation PDF
import pdfplumber

pdf_path = os.path.join(original_repo, 'documentation.pdf')

with pdfplumber.open(pdf_path) as pdf:
    original_doc_text = ""
    for page in pdf.pages:
        text = page.extract_text()
        if text:
            original_doc_text += text + "\n\n"

print("--- Original Documentation (PDF) ---")
print(original_doc_text)

--- Original Documentation (PDF) ---
Locating and Editing Factual Associations in GPT
KevinMeng∗ DavidBau∗ AlexAndonian YonatanBelinkov†
MITCSAIL NortheasternUniversity MITCSAIL Technion–IIT
Abstract
Weanalyzethestorageandrecalloffactualassociationsinautoregressivetrans-
formerlanguagemodels,findingevidencethattheseassociationscorrespondto
localized,directly-editablecomputations. Wefirstdevelopacausalintervention
foridentifyingneuronactivationsthataredecisiveinamodel’sfactualpredictions.
Thisrevealsadistinctsetofstepsinmiddle-layerfeed-forwardmodulesthatme-
diatefactualpredictionswhileprocessingsubjecttokens. Totestourhypothesis
thatthesecomputationscorrespondtofactualassociationrecall,wemodifyfeed-
forward weights to update specific factual associations using Rank-One Model
Editing(ROME).WefindthatROMEiseffectiveonastandardzero-shotrelation
extraction(zsRE)model-editingtask. WealsoevaluateROMEonanewdataset
ofdifficultcounterfactualassertions,onwhichitsimultaneouslymaintainsboth
specif